# Breakpoints y Criterio de Información Bayesiano (BIC)

> Since `piecewise-regression` calculates the BIC with random initialization, the choice of the **number of breakpoints** must be validated by performing the calculation multiple times (`n_trials`).

---

## Import libraries

In [1]:
import sys
import os
root = os.path.abspath('..')  
sys.path.append(root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from modules import load, plots, analysis
from modules.archive import processing

---

## Parameters

In [ ]:
# FILE_NAME: Name of the input CSV file containing raw profile measurements (e.g., depth vs conductivity).
FILE_NAME = 'BW9D_YSI_20230823_rowdy'
# INPUT_DIR: Folder (relative to project root) where input files are located.
INPUT_DIR = os.path.join(root, 'data', 'rawdy')
# FILE_PATH: Full path to the input file. If you set this explicitly, it takes precedence over INPUT_DIR/FILE_NAME.
FILE_PATH = os.path.join(INPUT_DIR, FILE_NAME + '.csv')

# Model search hyperparameters
# Maximum number of breakpoints (segments - 1). 
MAX_BREAKPOINTS = 10

# Number of random restarts / trials per model size to mitigate local minima.
N_TRIALS = 3

# Output directory for artifacts
OUTPUT_DIR = os.path.join(root, 'data', 'results')

# Column names in the input CSV (edit these if your file headers differ)
VP_NAME = 'Vertical Position [m]'           # independent variable (x)
SEC_NAME = 'Corrected sp Cond [uS/cm]'      # dependent variable (y)

# Toggle to persist results as JSON to OUTPUT_DIR
SAVE_BIC_RESULTS = True


---

## Load data

In [3]:
# Prefer FILE_PATH if set; otherwise, compose from INPUT_DIR and FILE_NAME
csv_path = FILE_PATH if os.path.isabs(FILE_PATH) or FILE_PATH else os.path.join(INPUT_DIR, FILE_NAME)
print(f"Reading data from: {csv_path}")
df = pd.read_csv(csv_path)

# Extract vectors
x = df[VP_NAME].to_numpy()
y = df[SEC_NAME].to_numpy()

Reading data from: c:\Users\Arhui\Desktop\proyectos\mar\freshwater_lens\data\rawdy\BW9D_YSI_20230823_rowdy.csv


---

## Calculate BIC

In [4]:
results = analysis.best_n_breakpoints(x, y, 
                    max_breakpoints=MAX_BREAKPOINTS, 
                    n_trials=N_TRIALS
                    )

Running fit with n_breakpoint = 0 . . 
Running fit with n_breakpoint = 1 . . 
Running fit with n_breakpoint = 2 . . 
Running fit with n_breakpoint = 3 . . 

                 Breakpoint Model Comparision Results                 
n_breakpoints            BIC    converged          RSS 
----------------------------------------------------------------------------------------------------
0                 7.7053e+04         True   4.7241e+11 
1                 7.5746e+04         True    3.435e+11 
2                 5.6499e+04         True   3.3227e+09 
3                 5.5077e+04         True   2.3496e+09 

Min BIC (Bayesian Information Criterion) suggests best model
Running fit with n_breakpoint = 0 . . 
Running fit with n_breakpoint = 1 . . 
Running fit with n_breakpoint = 2 . . 
Running fit with n_breakpoint = 3 . . 

                 Breakpoint Model Comparision Results                 
n_breakpoints            BIC    converged          RSS 
---------------------------------------------

---

## Save BIC results

> Calculating different piecewise linear fits is intensive; the information is stored in `JSON` format to avoid repeating this process.

In [5]:
# --- Save results to JSON (optional) ---
if SAVE_BIC_RESULTS:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    results_df = pd.DataFrame(results)
    out_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(FILE_NAME)[0]}_bic.json")
    results_df.to_json(out_path, indent=2)
    print(f"Saved results to: {out_path}")
else:
    print("Skipping save. Set SAVE_BIC_RESULTS = True to export results as JSON.")

Saved results to: c:\Users\Arhui\Desktop\proyectos\mar\freshwater_lens\data\results\BW9D_YSI_20230823_rowdy_bic.json
